In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

schema = "healthcare_project"

# -----------------------------
# Load Silver tables
# -----------------------------
patients = spark.table(f"{schema}.silver_patients")
encounters = spark.table(f"{schema}.silver_encounters")
conditions = spark.table(f"{schema}.silver_conditions")
procedures = spark.table(f"{schema}.silver_procedures")
medications = spark.table(f"{schema}.silver_medications")
providers = spark.table(f"{schema}.silver_providers")

# -----------------------------
# 1) Gold Patient Visit Timeline
# Window functions: row_number, lag
# Business use: visit sequencing, revisit gap analysis
# -----------------------------
patient_visit_window = Window.partitionBy("patient_id").orderBy("encounter_start")

gold_patient_visit_timeline = (
    encounters
    .withColumn("visit_sequence", F.row_number().over(patient_visit_window))
    .withColumn("previous_encounter_date", F.lag("encounter_date").over(patient_visit_window))
    .withColumn(
        "days_since_last_visit",
        F.datediff(F.col("encounter_date"), F.col("previous_encounter_date"))
    )
    .withColumn(
        "revisit_within_30_days",
        F.when(
            F.col("days_since_last_visit").isNotNull() & (F.col("days_since_last_visit") <= 30),
            1
        ).otherwise(0)
    )
    .select(
        "patient_id",
        "encounter_id",
        "encounter_start",
        "encounter_end",
        "encounter_date",
        "encounter_year",
        "encounter_month",
        "encounter_class",
        "encounter_description",
        "total_claim_cost",
        "length_of_stay_hours",
        "visit_sequence",
        "previous_encounter_date",
        "days_since_last_visit",
        "revisit_within_30_days"
    )
)

gold_patient_visit_timeline.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_patient_visit_timeline"
)

# -----------------------------
# 2) Gold Patient Cost Journey
# Window function: cumulative sum over patient timeline
# Business use: patient healthcare spend progression
# -----------------------------
patient_cost_window = (
    Window.partitionBy("patient_id")
    .orderBy("encounter_start")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

latest_encounter_window = Window.partitionBy("patient_id").orderBy(F.col("encounter_start").desc())

gold_patient_cost_journey = (
    encounters
    .withColumn(
        "cumulative_claim_cost",
        F.sum("total_claim_cost").over(patient_cost_window)
    )
    .withColumn(
        "latest_encounter_rank",
        F.row_number().over(latest_encounter_window)
    )
    .select(
        "patient_id",
        "encounter_id",
        "encounter_start",
        "encounter_date",
        "total_claim_cost",
        "cumulative_claim_cost",
        "latest_encounter_rank"
    )
)

gold_patient_cost_journey.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_patient_cost_journey"
)

# -----------------------------
# 3) Gold Provider Rankings
# Window function: dense_rank
# Business use: top providers by revenue and encounter volume
# -----------------------------
provider_agg = (
    encounters
    .join(
        providers.select("provider_id", "provider_name", "speciality", "city", "state"),
        "provider_id",
        "left"
    )
    .groupBy("provider_id", "provider_name", "speciality", "city", "state")
    .agg(
        F.count("*").alias("total_encounters"),
        F.sum("total_claim_cost").alias("total_revenue"),
        F.avg("total_claim_cost").alias("avg_revenue_per_encounter"),
        F.avg("length_of_stay_hours").alias("avg_length_of_stay_hours")
    )
)

provider_rank_window = Window.orderBy(F.desc("total_revenue"))

gold_provider_rankings = (
    provider_agg
    .withColumn("provider_revenue_rank", F.dense_rank().over(provider_rank_window))
    .orderBy("provider_revenue_rank", F.desc("total_encounters"))
)

gold_provider_rankings.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_provider_rankings"
)

# -----------------------------
# 4) Gold Frequent Revisit Patients
# Business use: patients with high short-term revisit behavior
# -----------------------------
gold_frequent_revisit_patients = (
    gold_patient_visit_timeline
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("total_visits"),
        F.sum("revisit_within_30_days").alias("revisits_within_30_days"),
        F.max("encounter_date").alias("last_visit_date")
    )
    .filter(F.col("revisits_within_30_days") > 0)
    .orderBy(F.desc("revisits_within_30_days"), F.desc("total_visits"))
)

gold_frequent_revisit_patients.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_frequent_revisit_patients"
)

# -----------------------------
# 5) Gold Diagnosis Trends
# Business use: top conditions by year
# -----------------------------
gold_diagnosis_trends = (
    conditions
    .join(
        encounters.select("encounter_id", "encounter_year", "encounter_month"),
        "encounter_id",
        "left"
    )
    .groupBy("encounter_year", "condition_description")
    .agg(
        F.count("*").alias("diagnosis_count")
    )
    .orderBy(F.desc("diagnosis_count"))
)

gold_diagnosis_trends.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_diagnosis_trends"
)

# -----------------------------
# 6) Gold Medication Usage by Age Group
# Business use: medication patterns across demographics
# -----------------------------
gold_medication_usage_by_age = (
    medications
    .join(
        patients.select("patient_id", "age", "age_group", "gender", "race", "ethnicity"),
        "patient_id",
        "left"
    )
    .groupBy("age_group", "medication_description")
    .agg(
        F.count("*").alias("usage_count"),
        F.avg("base_cost").alias("avg_medication_cost"),
        F.avg("total_cost").alias("avg_total_medication_cost")
    )
    .orderBy(F.desc("usage_count"))
)

gold_medication_usage_by_age.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_medication_usage_by_age"
)

# -----------------------------
# 7) Gold Procedure Utilization by Specialty
# Business use: most common procedures by provider specialty
# -----------------------------
gold_procedure_utilization_by_specialty = (
    procedures
    .join(
        encounters.select("encounter_id", "provider_id", "encounter_year"),
        "encounter_id",
        "left"
    )
    .join(
        providers.select("provider_id", "provider_name", "speciality"),
        "provider_id",
        "left"
    )
    .groupBy("speciality", "procedure_description")
    .agg(
        F.count("*").alias("procedure_count"),
        F.avg("base_cost").alias("avg_procedure_cost")
    )
    .orderBy(F.desc("procedure_count"))
)

gold_procedure_utilization_by_specialty.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_procedure_utilization_by_specialty"
)

# -----------------------------
# 8) Gold Patient Encounter Summary
# Business use: patient-level utilization and cost summary
# -----------------------------
gold_patient_encounter_summary = (
    encounters
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("total_encounters"),
        F.sum("total_claim_cost").alias("total_cost"),
        F.avg("total_claim_cost").alias("avg_cost_per_encounter"),
        F.max("encounter_date").alias("last_visit_date"),
        F.avg("length_of_stay_hours").alias("avg_length_of_stay_hours")
    )
    .join(
        patients.select("patient_id", "age", "age_group", "gender", "race", "ethnicity", "city", "state"),
        "patient_id",
        "left"
    )
)

gold_patient_encounter_summary.write.mode("overwrite").saveAsTable(
    f"{schema}.gold_patient_encounter_summary"
)

print("All upgraded Gold tables created successfully.")

All upgraded Gold tables created successfully.


In [0]:
gold_tables = [
    "gold_patient_visit_timeline",
    "gold_patient_cost_journey",
    "gold_provider_rankings",
    "gold_frequent_revisit_patients",
    "gold_diagnosis_trends",
    "gold_medication_usage_by_age",
    "gold_procedure_utilization_by_specialty",
    "gold_patient_encounter_summary"
]

for table in gold_tables:
    print(f"\nPreview: {table}")
    display(spark.table(f"{schema}.{table}").limit(10))


Preview: gold_patient_visit_timeline


patient_id,encounter_id,encounter_start,encounter_end,encounter_date,encounter_year,encounter_month,encounter_class,encounter_description,total_claim_cost,length_of_stay_hours,visit_sequence,previous_encounter_date,days_since_last_visit,revisit_within_30_days
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-5e1c-d66184ba9891,1938-06-04T23:15:24Z,1938-06-06T09:57:09Z,1938-06-04,1938,6,inpatient,Encounter For Problem (procedure),8642.82,34.69583333333333,1,null,null,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-7672-ac36499621fd,1944-07-29T23:15:24Z,1944-07-30T00:10:43Z,1944-07-29,1944,7,wellness,General Examination Of Patient (procedure),568.2,0.9219444444444445,2,1938-06-04,2247,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-a7b1-3f740b650b93,1953-08-23T23:15:24Z,1953-08-23T23:30:24Z,1953-08-23,1953,8,ambulatory,Prenatal Visit (regime/therapy),1005.38,0.25,3,1944-07-29,3312,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-05d0-96629df0a370,1957-08-17T23:15:24Z,1957-08-17T23:51:22Z,1957-08-17,1957,8,wellness,General Examination Of Patient (procedure),778.78,0.5994444444444444,4,1953-08-23,1455,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-882b-98a6360e693f,1962-04-11T00:08:02Z,1962-04-12T00:08:02Z,1962-04-11,1962,4,inpatient,Admission To Surgical Department (procedure),3817.64,24.0,5,1957-08-17,1698,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-179e-277daea22d2e,1963-08-24T23:15:24Z,1963-08-24T23:58:39Z,1963-08-24,1963,8,wellness,General Examination Of Patient (procedure),778.78,0.7208333333333333,6,1962-04-11,500,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-0ca5-f891ef8b21be,1966-06-11T23:15:24Z,1966-06-12T00:08:52Z,1966-06-11,1966,6,wellness,General Examination Of Patient (procedure),778.78,0.8911111111111111,7,1963-08-24,1022,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-9368-effe77927906,1968-06-15T23:15:24Z,1968-06-15T23:47:13Z,1968-06-15,1968,6,wellness,General Examination Of Patient (procedure),778.78,0.5302777777777777,8,1966-06-11,735,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-78f5-08b359163234,1970-06-20T23:15:24Z,1970-06-20T23:46:37Z,1970-06-20,1970,6,wellness,General Examination Of Patient (procedure),1567.0,0.5202777777777777,9,1968-06-15,735,0
001d3bfc-937a-321c-7b91-a88e53e52115,001d3bfc-937a-321c-1fc9-faae782308f1,1972-06-24T23:15:24Z,1972-06-24T23:58:52Z,1972-06-24,1972,6,wellness,General Examination Of Patient (procedure),853.36,0.7244444444444444,10,1970-06-20,735,0



Preview: gold_patient_cost_journey


patient_id,encounter_id,encounter_start,encounter_date,total_claim_cost,cumulative_claim_cost,latest_encounter_rank
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-3ec3-77cae1ec6ba6,2026-03-05T19:57:09Z,2026-03-05,881.81,241326.92999999993,1
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-f9d2-592a3777f5ea,2025-12-18T19:57:09Z,2025-12-18,3105.35,240445.11999999994,2
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-9b32-eb53d761d49a,2025-12-11T19:57:09Z,2025-12-11,637.47,237339.76999999993,3
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-5f16-121ceae8e537,2025-12-04T19:57:09Z,2025-12-04,919.64,236702.29999999993,4
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-97f2-a923665adfef,2025-11-20T19:57:09Z,2025-11-20,2673.95,235782.65999999992,5
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-0138-e1f253a7c421,2025-11-06T19:57:09Z,2025-11-06,666.11,233108.7099999999,6
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-6659-f8b6f15c5a38,2025-10-16T19:57:09Z,2025-10-16,3105.35,232442.59999999992,7
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-9e0c-8e30e510b4ad,2025-10-02T19:57:09Z,2025-10-02,881.81,229337.2499999999,8
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-cb3b-afc75fe6307f,2025-09-18T19:57:09Z,2025-09-18,3105.35,228455.43999999992,9
00022181-2aac-eb42-7733-1b31563f7f23,00022181-2aac-eb42-0f95-0b72a5782075,2025-09-04T19:57:09Z,2025-09-04,1433.33,225350.0899999999,10



Preview: gold_provider_rankings


...

age_group,medication_description,usage_count,avg_medication_cost,avg_total_medication_cost
65+,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],65560,29.74097879805125,29.74097879805125
65+,Insulin Isophane Human 70 Unt/ml / Insulin Regular Human 30 Unt/ml Injectable Suspension [humulin],38676,246.56732211189714,423.63377391664926
65+,Lisinopril 10 Mg Oral Tablet,33691,0.9029749784808064,1.3577994123058368
65+,Hydrochlorothiazide 25 Mg Oral Tablet,30739,0.5556859364324866,0.8175324506326845
65+,Amlodipine 2.5 Mg Oral Tablet,26049,1.043712618526546,1.5558612614687162
51-65,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],25831,29.745556889009965,29.745556889009965
65+,Cisplatin 50 Mg Injection,17965,222.75275535764052,222.75275535764052
65+,Paclitaxel 100 Mg Injection,15336,1.7627516953572797,1.7627516953572797
51-65,Lisinopril 10 Mg Oral Tablet,14946,0.9086939649404162,1.7243817743877552
65+,Sodium Fluoride 0.0272 Mg/mg Oral Gel,13224,129.94000000000213,129.94000000000213



Preview: gold_procedure_utilization_by_specialty


speciality,procedure_description,procedure_count,avg_procedure_cost
General Practice,Depression Screening (procedure),209376,431.39999999983485
General Practice,Assessment Of Health And Social Care Needs (procedure),126206,431.40000000023207
General Practice,Renal Dialysis (procedure),122021,834.1382547267895
General Practice,Medication Reconciliation (procedure),87548,418.89525608810624
General Practice,Assessment Of Substance Use (procedure),83433,431.40000000016266
General Practice,Patient Referral For Dental Care (procedure),74056,431.4000000001369
General Practice,Dental Consultation And Report (procedure),71903,431.40000000012975
General Practice,Oral Health Education (procedure),69801,431.40000000012265
General Practice,Removal Of Supragingival Plaque And Calculus From All Teeth Using Dental Instrument (procedure),69383,431.4000000001212
General Practice,Removal Of Subgingival Plaque And Calculus From All Teeth Using Dental Instrument (procedure),69383,431.4000000001212



Preview: gold_patient_encounter_summary


patient_id,total_encounters,total_cost,avg_cost_per_encounter,last_visit_date,avg_length_of_stay_hours,age,age_group,gender,race,ethnicity,city,state
a99869df-ac9c-71a5-7e7f-3979444fe082,736,784467.289999999,1065.8522961956508,2022-03-09,3.3696335295893713,57,51-65,F,White,Nonhispanic,Agawam,MASSACHUSETTS
b99905b1-719f-6041-05cd-53b37061f5a3,74,150652.88999999996,2035.8498648648642,2026-04-01,1.8085848348348348,49,36-50,F,White,Nonhispanic,Bourne,MASSACHUSETTS
f57d64c0-493a-59d5-e135-ebc7a70a4603,66,180159.97,2729.696515151515,2026-01-18,0.8038215488215488,45,36-50,F,White,Nonhispanic,Reading,MASSACHUSETTS
a0b4d474-d114-288e-bccd-dab98470aa42,130,192423.69999999998,1480.1823076923076,2025-10-19,4.822502136752136,98,65+,F,Black,Nonhispanic,Norwood,MASSACHUSETTS
cd2c1063-99ad-9b61-d07b-99085b465da5,36,201327.91999999998,5592.442222222222,2026-04-02,0.719537037037037,34,18-35,F,White,Nonhispanic,Quincy,MASSACHUSETTS
8aa05347-46bb-7358-ab23-f885756e1779,89,82784.81000000001,930.1664044943822,2026-03-22,3.138392634207241,64,51-65,M,Hawaiian,Nonhispanic,Pittsfield,MASSACHUSETTS
32d74a66-ce31-15d5-9734-0d644977aaa3,28,59138.159999999996,2112.077142857143,2026-03-27,1.9800099206349207,48,36-50,F,White,Nonhispanic,Tewksbury,MASSACHUSETTS
353637e7-579a-69a5-380f-bc43be1f6226,20,14997.9,749.895,2026-02-13,0.4842638888888889,6,Under 18,M,Asian,Nonhispanic,Medford,MASSACHUSETTS
8c83996e-773a-3d71-6080-e6e01467ac29,15,15777.699999999997,1051.8466666666666,2026-02-26,0.3,3,Under 18,F,White,Nonhispanic,Seekonk,MASSACHUSETTS
6d40ecd9-da31-1100-e15c-115af8fac93f,14,54978.950000000004,3927.0678571428575,1993-12-23,16.39263888888889,73,65+,M,White,Nonhispanic,Ludlow,MASSACHUSETTS


In [0]:
for table in gold_tables:
    print(f"\nCount check: {table}")
    spark.sql(f"SELECT COUNT(*) AS row_count FROM {schema}.{table}").show()


Count check: gold_patient_visit_timeline
+---------+
|row_count|
+---------+
|   742364|
+---------+


Count check: gold_patient_cost_journey
+---------+
|row_count|
+---------+
|   742364|
+---------+


Count check: gold_provider_rankings
+---------+
|row_count|
+---------+
|     1833|
+---------+


Count check: gold_frequent_revisit_patients
+---------+
|row_count|
+---------+
|    12537|
+---------+


Count check: gold_diagnosis_trends
+---------+
|row_count|
+---------+
|    11438|
+---------+


Count check: gold_medication_usage_by_age
+---------+
|row_count|
+---------+
|     1048|
+---------+


Count check: gold_procedure_utilization_by_specialty
+---------+
|row_count|
+---------+
|      410|
+---------+


Count check: gold_patient_encounter_summary
+---------+
|row_count|
+---------+
|    12736|
+---------+

